<a href="https://colab.research.google.com/github/jgeiger81/sbml/blob/main/5.5_ML_Models/JasonG_MLE_MiniProject_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mini Project: Build a Machine Learning Model

## Predict Total Fare on the NYC Taxi Dataset

Welcome to the NYC Taxi Fare Prediction project! In this Colab, we will continue using the NYC Taxi Dataset to predict the fare amount for taxi rides using a subset of available features. We will go through three main stages: building a baseline model, creating a full model, and performing hyperparameter tuning to enhance our predictions.

Now that you've completed exploratory data analysis on this dataset you should have a good understanding of the feature space.

## Project Objectives

The primary objectives of this project are as follows:

Baseline Model: We will start by building a simple baseline model to establish a benchmark for our predictions. This model will serve as a starting point to compare the performance of our subsequent models.

Full Model: Next, we will develop a more comprehensive model that leverages machine learning techniques to improve prediction accuracy. We will use Scikit-Learn's model pipeline to build a framework that enables rapid experimentation.

Hyperparameter Tuning: Lastly, we will optimize our full model by fine-tuning its hyperparameters. By systematically adjusting the parameters that control model behavior, we aim to achieve the best possible performance for our prediction task.

In [28]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_absolute_error

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

Load the NYC taxi dataset into a Pandas DataFrame and do a few basic checks to ensure the data is loaded properly. Note, there are several months of data that can be used. For simplicity, use the Yellow Taxi 2022-01 parquet file [here](https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2022-01.parquet). Here are your tasks:

  1. Load the `yellow_tripdata_2022-01.parquet` file into Pandas.
  2. Print the first 5 rows of data.
  3. Drop any rows of data that contain NULL values.
  4. Create a new feature, 'trip_duration' that captures the duration of the trip in minutes.
  5. Create a varible named 'target_variable' to store the name of the thing we're trying to predict, 'total_amount'.
  6. Create a list called 'feature_cols' containing the feature names that we'll be using to predict our target variable. The list should contain 'VendorID', 'trip_distance', 'payment_type', 'PULocationID', 'DOLocationID', and 'trip_duration'.

In [38]:
# Load the dataset into a pandas DataFrame (from https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page)

df = pd.read_parquet('yellow_tripdata_2022-01.parquet')
#df = pd.read_parquet('https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2022-01.parquet')


**Interesting we are going back to this very dirty data set.**

In [3]:
# Display the shape of the dataset
# Woah, big guy
df.shape

(2463931, 19)

**Not crazy big**

In [29]:
# Display the first few rows of the dataset
# default pandas settings show first 5 and last 5 rows
df

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee
0,1,2022-01-01 00:35:40,2022-01-01 00:53:29,2.0,3.80,1.0,N,142,236,1,14.50,3.0,0.5,3.65,0.0,0.3,21.95,2.5,0.0
1,1,2022-01-01 00:33:43,2022-01-01 00:42:07,1.0,2.10,1.0,N,236,42,1,8.00,0.5,0.5,4.00,0.0,0.3,13.30,0.0,0.0
2,2,2022-01-01 00:53:21,2022-01-01 01:02:19,1.0,0.97,1.0,N,166,166,1,7.50,0.5,0.5,1.76,0.0,0.3,10.56,0.0,0.0
3,2,2022-01-01 00:25:21,2022-01-01 00:35:23,1.0,1.09,1.0,N,114,68,2,8.00,0.5,0.5,0.00,0.0,0.3,11.80,2.5,0.0
4,2,2022-01-01 00:36:48,2022-01-01 01:14:20,1.0,4.30,1.0,N,68,163,1,23.50,0.5,0.5,3.00,0.0,0.3,30.30,2.5,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2463926,2,2022-01-31 23:36:53,2022-01-31 23:42:51,NaN,1.32,NaN,None,90,170,0,8.00,0.0,0.5,2.39,0.0,0.3,13.69,NaN,NaN
2463927,2,2022-01-31 23:44:22,2022-01-31 23:55:01,NaN,4.19,NaN,None,107,75,0,16.80,0.0,0.5,4.35,0.0,0.3,24.45,NaN,NaN
2463928,2,2022-01-31 23:39:00,2022-01-31 23:50:00,NaN,2.10,NaN,None,113,246,0,11.22,0.0,0.5,2.00,0.0,0.3,16.52,NaN,NaN
2463929,2,2022-01-31 23:36:42,2022-01-31 23:48:45,NaN,2.92,NaN,None,148,164,0,12.40,0.0,0.5,0.00,0.0,0.3,15.70,NaN,NaN


In [30]:
# Check for missing values
#look for null values, and count them up - by column
df.isnull().sum()

VendorID                     0
tpep_pickup_datetime         0
tpep_dropoff_datetime        0
passenger_count          71503
trip_distance                0
RatecodeID               71503
store_and_fwd_flag       71503
PULocationID                 0
DOLocationID                 0
payment_type                 0
fare_amount                  0
extra                        0
mta_tax                      0
tip_amount                   0
tolls_amount                 0
improvement_surcharge        0
total_amount                 0
congestion_surcharge     71503
airport_fee              71503
dtype: int64

In [31]:
# Drop rows with missing values.
df = df.dropna(axis='rows').copy() #target rows, not columns
df.shape ## 71,503 less rows, as predicted

(2392428, 19)

In [32]:
df.isnull().sum()

VendorID                 0
tpep_pickup_datetime     0
tpep_dropoff_datetime    0
passenger_count          0
trip_distance            0
RatecodeID               0
store_and_fwd_flag       0
PULocationID             0
DOLocationID             0
payment_type             0
fare_amount              0
extra                    0
mta_tax                  0
tip_amount               0
tolls_amount             0
improvement_surcharge    0
total_amount             0
congestion_surcharge     0
airport_fee              0
dtype: int64

**Cleaned**

In [33]:
# Create new feature, 'trip_duration'.

df["trip_duration"] = round((df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]).dt.total_seconds() / 60)
df["trip_duration"]


0          18.0
1           8.0
2           9.0
3          10.0
4          38.0
           ... 
2392423     8.0
2392424     4.0
2392425     8.0
2392426     8.0
2392427    12.0
Name: trip_duration, Length: 2392428, dtype: float64

In [34]:
# Create a list called feature_col to store column names
feature_col = ['VendorID', 'trip_distance', 'payment_type', 'PULocationID', 'DOLocationID', 'trip_duration']


Splitting a dataset into training and test sets is a crucial step in machine learning model development. It allows us to evaluate the performance and generalization ability of our models accurately. The training set is used to train the model, while the test set serves as an independent sample for evaluating its performance.

1. **Model Training**: The training set is used to fit the model, allowing it to learn the underlying patterns and relationships between the features and the target variable. By exposing the model to a diverse range of examples in the training set, it can capture the underlying structure of the data.

2. **Model Evaluation**: The test set, which is independent of the training set, is crucial for evaluating how well the trained model generalizes to unseen data. It provides an unbiased assessment of the model's performance on new instances. By measuring the model's accuracy, precision, recall, or other evaluation metrics on the test set, we can estimate how well the model will perform on unseen data.

3. **Preventing Overfitting**: Overfitting occurs when a model learns the training data's noise and idiosyncrasies instead of the underlying patterns. By evaluating the model on the test set, we can identify if the model is overfitting. If the model performs significantly worse on the test set compared to the training set, it indicates overfitting. In such cases, we might need to adjust the model, feature selection, or regularization techniques to improve generalization.

4. **Hyperparameter Tuning**: Splitting the dataset allows us to perform hyperparameter tuning on the model. Hyperparameters are configuration settings that control the learning process, such as learning rate, regularization strength, or the number of hidden layers in a neural network. By using a validation set (often created from a portion of the training set), we can iteratively adjust the hyperparameters and select the best combination that maximizes the model's performance on the validation set. The final evaluation on the test set provides an unbiased estimate of the model's performance.

By splitting the dataset into training and test sets, we can ensure that our models are both well-trained and accurately evaluated. This separation helps us understand how the model will perform on new, unseen data, which is critical for assessing its effectiveness and making informed decisions about its deployment.

Here is your task:

  1. Use Scikit-Learn's [train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) to split the data into training and test sets. Don't forget to set the random state.

In [35]:
# Split dataset into training and test sets

X = df[feature_col]
y = df['total_amount']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


The importance of a baseline model, even if it uses a simple strategy like always predicting the mean, cannot be understated. Here's why a baseline model is valuable:

1. **Performance Comparison**: A baseline model serves as a reference point for evaluating the performance of more sophisticated models. By establishing a simple yet reasonable baseline, we can determine whether our advanced models offer any significant improvement over this basic approach. It helps us set realistic expectations and gauge the effectiveness of our efforts.

2. **Model Complexity**: A baseline model provides insight into the complexity required to solve the prediction task. If a simple strategy like predicting the median performs reasonably well, it suggests that the problem might not necessitate complex modeling techniques. Conversely, if the baseline model performs poorly, it indicates the presence of more intricate patterns that need to be captured by more sophisticated models.

3. **Minimum Performance Requirement**: A baseline model can establish a minimum performance requirement for a predictive task. If we cannot outperform the baseline, it suggests that our models have failed to capture even the most fundamental relationships within the data. In such cases, we may need to revisit our data preprocessing steps, feature engineering techniques, or consider other external factors affecting the task.

4. **Identifying Data Issues**: A baseline model can help identify potential issues within the dataset. If the baseline model performs poorly, it may indicate problems like missing values, outliers, or data inconsistencies. These issues can be further investigated and resolved to improve the overall model performance.

While a baseline model like always predicting the median may not offer the highest prediction accuracy, its importance lies in its role as a starting point for model development and evaluation. It provides a solid foundation for comparing and assessing the performance of more complex models, ensuring that any improvements made are meaningful and significant.

Here is your task:

  1. Create a model that always predicts the mean total fare of the training dataset. Use Scikit-Learn's [mean_absolute_error](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_absolute_error.html) to evaluate this model. Is it any good?

In [36]:
# Create a baseline for mean absolute error of total amount: mean_absolute_error

#Create an array of length of the test set, containing only the mean value of the training set.
baseline_prediction = np.full(len(y_test), y_train.mean())
baseline_prediction


array([19.06196355, 19.06196355, 19.06196355, ..., 19.06196355,
       19.06196355, 19.06196355], shape=(478486,))

In [12]:
# Calculate the baseline MAE
mae_baseline = mean_absolute_error(y_test, baseline_prediction)
mae_baseline
# print(f"Baseline MAE: ${baseline_mae:.2f}")

9.198227928516678

**So, this means that our baseline prediciton is off by 9.19, on average. Which is not great. We'll have to beat that later.**

If outliers are high, wouldn't we want to use median?

With a baseline metric in place, we can try to build a machine learning model. Obviously, if the model can't beat the baseline then there are some major issues to be resolved.

It's always a good idea to start with a simple machine learning model, like linear regression, and build upon it if necessary.

Here are your tasks:

  1. Use Scikit-Learn's [ColumnTransformer](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html) to preprocess the categorical and continuous features independently. Apply the [StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) to the continuous columns and [OneHotEncoder](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html) to the categorical columns.

  One-hot encoding is a popular technique used to represent categorical variables numerically in machine learning models. It transforms categorical features into a binary vector representation, where each category is represented by a binary column. Here's an explanation of one-hot encoding:

  When working with categorical variables, such as colors (e.g., red, blue, green) or vehicle types (e.g., car, truck, motorcycle), machine learning algorithms often require numerical inputs. However, directly assigning numerical values to categories can introduce unintended relationships or orderings between them. For example, assigning the values 0, 1, and 2 to the categories red, blue, and green may imply a sequential relationship, which is not desired.

  One-hot encoding solves this problem by creating new binary columns, equal to the number of unique categories in the original feature. Each binary column represents a specific category and takes a value of 1 if the data point belongs to that category, and 0 otherwise. This encoding ensures that no implicit ordering or relationship exists between the categories.

  2. Integrate the preprocessor in the previous step with Scikit-Learn's [LinearRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html) model using a [Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html).

  3. Train the pipeline on the training data.

  4. Evaluate the model using mean absolute error as a metric on the test data. Does the model beat the baseline?


In [13]:
# Use Scikit-Learn's ColumnTransformer to preprocess the categorical and
# continuous features independently.

#I Thought all of our column were all continuous, so let's look

#For future reuse. Honestly, you'd think there was tooling in Pandas to take
#it's best guess as to whether a feature was categorical or continuous, but
#I did not find anything.
def inspectData(train):
  print("Quick summary of features:")
  for col in feature_col:
      print(f"\n{col}:")
      print(f"  Data type: {train[col].dtype}")
      print(f"  Unique values: {train[col].nunique()}")
      print(f"  Sample values: {train[col].head(3).tolist()}")

inspectData(X_train)

Quick summary of features:

VendorID:
  Data type: int64
  Unique values: 2
  Sample values: [2, 2, 2]

trip_distance:
  Data type: float64
  Unique values: 4003
  Sample values: [2.57, 2.83, 3.26]

payment_type:
  Data type: int64
  Unique values: 5
  Sample values: [2, 1, 2]

PULocationID:
  Data type: int64
  Unique values: 256
  Sample values: [79, 246, 52]

DOLocationID:
  Data type: int64
  Unique values: 261
  Sample values: [141, 237, 232]

trip_duration:
  Data type: float64
  Unique values: 606
  Sample values: [21.0, 20.0, 15.0]


**Ha, Nope!**
- Categorical: VendorId, payment_type, PULocatonId, DOLocationId
- Continuous: trip_distance, trip_duration

In [14]:
# #Let's manually define these two here
# categorical_feat = ['VendorID', 'payment_type', 'PULocationID', 'DOLocationID']
# continuous_feat = ['trip_distance', 'trip_duration']

# # Grab the categorical and continuous features separately
# X_train_continuous = X_train[continuous_feat]
# X_test_continuous = X_test[continuous_feat]
# X_train_categorical = X_train[categorical_feat]
# X_test_categorical = X_test[categorical_feat]

# # Scale the continuous ones
# scaler = StandardScaler()
# X_train_continuous_scaled = scaler.fit_transform(X_train_continuous)
# X_test_continuous_scaled = scaler.transform(X_test_continuous)  # Use same scaler!

# # Encode categorical features

# # Combine everything back together

Wait.... Need to use a pipeline. Shoot.

In [15]:
# Create a pipeline object containing the column transformations and regression
# model.

#Let's manually define these two here
categorical_feat = ['VendorID', 'payment_type', 'PULocationID', 'DOLocationID']
continuous_feat = ['trip_distance', 'trip_duration']

# Now to create the ColumnTransformers
preprocessor = ColumnTransformer( #curious what naming coventions exist here.
    transformers=[
        ('num', StandardScaler(), continuous_feat),  # Scale continuous features
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_feat)  # Encode categorical
    ])

# Create the full pipeline
pipeline_lr = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])
pipeline_lr

,steps,"[('preprocessor', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


**Pretty!**

In [16]:
# Fit the pipeline on the training data.
pipeline_lr.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [17]:
# Make predictions on the test data.
y_pred_lr = pipeline_lr.predict(X_test)
y_pred_lr

array([14.44221045, 15.5660661 , 19.07590337, ..., 16.36928774,
       17.31309604, 11.29388392], shape=(478486,))

In [18]:
#Get this Model's mean absolute error to see how it preforms against baseline
mae_linear = mean_absolute_error(y_test, y_pred_lr)
mae_linear_base_diff = abs(mae_baseline - mae_linear)

#Print them out all nice
print(f"Linear Regression MAE: ${mae_linear:.2f} vs Baseline ${mae_baseline:.2f}")
print(f"Improvement: ${mae_linear_base_diff:.2f} --> {((mae_linear_base_diff)/mae_baseline * 100):.1f}% better")

Linear Regression MAE: $3.39 vs Baseline $9.20
Improvement: $5.81 --> 63.2% better


Random Forest Regression and Linear Regression are two commonly used regression algorithms, each with its own advantages and suitability for different scenarios. Random Forest Regression offers several advantages over Linear Regression, including:

1. **Non-linearity**: Random Forest Regressor is capable of capturing non-linear relationships between features and the target variable. In contrast, Linear Regression assumes a linear relationship between the features and the target. When faced with non-linear relationships or complex feature interactions, Random Forest Regressor can provide more accurate predictions.

2. **Robustness to Outliers**: Random Forest Regressor is generally more robust to outliers compared to Linear Regression. Outliers can disproportionately impact the coefficients and predictions of Linear Regression models. However, as an ensemble of decision trees, Random Forest Regressor can mitigate the effect of outliers by averaging predictions from multiple trees.

3. **Feature Importance**: Random Forest Regressor provides a measure of feature importance, which helps identify the most influential features for making predictions. This information is useful for feature selection, understanding the underlying relationships in the data, and gaining insights into the problem domain. Unlike Linear Regression, which provides coefficient values indicating the direction and magnitude of relationships, Random Forest Regressor explicitly highlights feature importance.

4. **Handling of Categorical Variables**: Random Forest Regressor can effectively handle categorical variables without requiring pre-processing steps like one-hot encoding. It can directly incorporate categorical variables into the model, making it more convenient when working with mixed data types. In contrast, Linear Regression often requires categorical variables to be encoded or transformed before use.

5. **Handling of High-Dimensional Data**: Random Forest Regressor can handle datasets with a large number of features (high dimensionality) by automatically selecting subsets of features during the construction of individual decision trees. This reduces the risk of overfitting, which is a concern with Linear Regression when dealing with high-dimensional data.

6. **Resistance to Multicollinearity**: Random Forest Regressor is less affected by multicollinearity, which occurs when predictor variables are highly correlated. In Linear Regression, highly correlated features can lead to unstable coefficient estimates, making it challenging to interpret the individual effects of each feature. Random Forest Regressor, as an ensemble approach, is less impacted by multicollinearity because each tree is built independently.

Here are your tasks:

  1. Build a Random Forest Regressor model using Scikit-Learn's [RandomForestRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html) and train it on the train data.

  2. Evaluate the performance of the model on the test data using mean absolute error as a metric. Mess around with various input parameter configurations to see how they affect the model. Can you beat the performance of the linear regression model?

In [19]:
# Build random forest regressor model
#Second Verse, (almost the) same as the first, and the preprocessor is already made

# Create the Random Forest pipeline, #going with 100 estimators here.
pipeline_rf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=10, max_depth=2, random_state=42, n_jobs=-1))
])

# Train the  model
pipeline_rf.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [20]:
# Make predictions on the test data
y_pred_rf = pipeline_rf.predict(X_test)
print(y_pred_rf)

[13.83091589 13.83091589 18.75074769 ... 13.83091589 13.83091589
 13.83091589]


In [21]:
# Let's see how this all compares
#Get this Model's mean absolute error to see how it preforms against baseline
mae_rf = mean_absolute_error(y_test, y_pred_rf)
mae_rf_base_diff = abs(mae_baseline - mae_rf)

mae_rf_lr_diff = mae_linear - mae_rf

#Print them out all nice and pretty
print(f"Baseline MAE: ${mae_baseline:.2f}")
print(f"Linear Regression MAE: ${mae_linear:.2f}")
print(f"Random Forest MAE: ${mae_rf:.2f}")

print(f"LR Improvement over Baseline: ${mae_linear_base_diff:.2f} --> {((mae_linear_base_diff)/mae_baseline * 100):.1f}% better")
print(f"RF Improvement over Baseline: ${mae_rf_base_diff:.2f} --> {((mae_rf)/mae_baseline * 100):.1f}% better")
print(f"RF Improvement over Linear Regression: ${mae_rf_lr_diff:.2f} --> {((mae_rf)/mae_linear * 100):.1f}% better")

Baseline MAE: $9.20
Linear Regression MAE: $3.39
Random Forest MAE: $3.90
LR Improvement over Baseline: $5.81 --> 63.2% better
RF Improvement over Baseline: $5.29 --> 42.4% better
RF Improvement over Linear Regression: $-0.52 --> 115.3% better


Hyperparameter tuning plays a critical role in machine learning model development. It involves selecting the optimal values for the hyperparameters, which are configuration settings that control the behavior of the learning algorithm. Here's why hyperparameter tuning is so important in ML:

1. **Optimizing Model Performance**: The choice of hyperparameters can significantly impact the model's performance. By fine-tuning the hyperparameters, we can improve the model's accuracy, precision, recall, or other performance metrics. It helps to extract the maximum predictive power from the chosen algorithm and ensures that the model is well-suited to the specific problem at hand.

2. **Avoiding Overfitting and Underfitting**: Hyperparameter tuning helps strike a balance between overfitting and underfitting.

3. **Exploring Model Complexity**: Hyperparameter tuning enables us to explore the complexity of the model. For instance, in algorithms like decision trees or neural networks, we can adjust the number of layers, the number of neurons, or the maximum depth of the tree. By systematically modifying these hyperparameters, we can understand how different levels of complexity impact the model's performance and find the right balance between simplicity and complexity.

Note, there are multiple approaches to hyperparemeter tuning.  

While grid search is the easiest to understand and implement there are many advantages of Bayesian search over grid search for hyperparameter tuning:

1. **Efficiency**: Bayesian search is generally more efficient than grid search. Grid search explores all possible combinations of hyperparameter values, which can be computationally expensive and time-consuming, especially when dealing with a large number of hyperparameters or a wide range of values. Bayesian search, on the other hand, intelligently selects the next hyperparameter configuration to evaluate based on the results of previous evaluations. It focuses on areas of the hyperparameter space that are more likely to yield better performance, reducing the number of evaluations needed.

2. **Flexibility**: Bayesian search is flexible in handling continuous and discrete hyperparameters. It can handle both types of hyperparameters naturally and effectively. In contrast, grid search is more suitable for discrete hyperparameters but may struggle with continuous ones, as it requires discretization or defining a finite set of values to search over.

3. **Adaptive Search**: Bayesian search adapts its search strategy based on the results of previous evaluations. It maintains a probability distribution over the hyperparameter space, updating it with each evaluation. This allows it to dynamically allocate more evaluations to promising regions and explore unexplored areas. In contrast, grid search follows a fixed and predefined search grid, regardless of the results of previous evaluations.

4. **Better Convergence**: Bayesian search has the potential to converge to the optimal hyperparameter configuration more quickly.

Here are your tasks:

  1. Perform a grid-search on a Random Forest Regressor model. Only search the space for the parameters 'n_estimators', 'max_depth', and 'min_samples_split'. Note, this can take some time to run. Make sure you set reasonable boundaries for the search space. Use Scikit-Learn's [GridSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html) method.

  2. After you've identified the best parameters, train a random forest regression model using these parameters on the full training data.

  3. Evaluate the model from the previous step using the test data. How does your model perform?

In [22]:
# Define the hyperparameters to tune.
# We're using Random Forest Regressor and asked to tune n_estimators, max_depth, and min_samples_split.
#We use "regressor__" prefix b/c that is how we address the regressor within the pipeline object
param_grid = {
    'regressor__n_estimators': [25, 50, 100],
    'regressor__max_depth': [1, 2, 3],
    'regressor__min_samples_split': [2,3]
}


In [23]:
# Perform grid search to find the best hyperparameters. This could take a while.

# Create GridSearchCV
grid_search_rf = GridSearchCV(
    pipeline_rf, #Already exists, re-use it.
    param_grid,
    cv=3,
    scoring='neg_mean_absolute_error', #meaning, the smallest MAE
    n_jobs=13,
    verbose=2
)


In [24]:
# Get the best model and its parameters.

# Fit the grid search
print("Starting hyperparameter tuning...")
grid_search_rf.fit(X_train, y_train)

Starting hyperparameter tuning...
Fitting 3 folds for each of 18 candidates, totalling 54 fits


KeyboardInterrupt: 

In [33]:
# Display the best hyperparams that we found for the best MAE
print("Best Hyperparams:")
print(grid_search_rf.best_params_)

Best Hyperparams:
{'regressor__max_depth': 3, 'regressor__min_samples_split': 2, 'regressor__n_estimators': 100}


In [34]:
# Fit the best classifier on the training data.

# Well, we already have a best fit Classifier, we just need to grab it
pipeline_best_rf = grid_search_rf.best_estimator_

In [35]:
# Make predictions on the test data
y_pred_best = pipeline_best_rf.predict(X_test)
mae_best_rf = mean_absolute_error(y_test, y_pred_best)

# Evaluate vs previous models w/o % improvement as that was starting to get messy
#and the values are easy to see and compare.
print(f"Baseline MAE:              ${mae_baseline:.2f}")
print(f"Linear Regression MAE:     ${mae_linear:.2f}")
print(f"Random Forest MAE:         ${mae_rf:.2f}")
print(f"Tuned Random Forest MAE:   ${mae_best_rf:.2f}")

Baseline MAE:              $9.20
Linear Regression MAE:     $3.39
Random Forest MAE:         $3.90
Tuned Random Forest MAE:   $3.32


Woo hoo, 7 cents better on average. Makes me think I am doing something wrong here with my hyperparam ranges. We can't get any better than that?

# Conclusions at the end of this assignment

First of all, I like the concept of just building tons of models across your hyperparameter space and training and cross validating them on the training data set to find out what is best.  It's also clear that GridSearch is painfully slow, quickly. At least in Collab with this dataset size. I switched to my local machine and pycharm to help things along. Even then...

It certainly drives home the point that Random, or Bayesian would be a better way to go about this on a limited CPU budget here.

Actually, I think I can try them out quick.

In [37]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

# Using broader ranges since random search samples randomly
    # 'regressor__n_estimators': [25, 50, 100],
    # 'regressor__max_depth': [1, 2, 3],
    # 'regressor__min_samples_split': [2,3]
param_grid_random = {
    'regressor__n_estimators': list(range(25, 150, 25)),  # integers from 25-150 step 25
    'regressor__max_depth': list(range(1, 5, 1)),  # Samples integers from 1-5
    'regressor__min_samples_split': randint(2, 3, 4)  # Samples integers from 2-4
}

print(param_grid_random)

# Create RandomizedSearchCV with new broader search area, limited to 20
random_search_rf = RandomizedSearchCV(
    pipeline_rf,
    param_grid_random, #use the same param grid from before, this way it should just be faster.
    n_iter=20,  # Number of random combinations to try. 20 ok? Who knows.
    cv=3,
    scoring='neg_mean_absolute_error',
    n_jobs=13,
    verbose=2,
    random_state=42
)

# Fit it
print("Starting Random Search hyperparameter tuning...")
random_search_rf.fit(X_train, y_train)

# Get best parames and the score
print(f"\nBest parameters: {random_search_rf.best_params_}")
print(f"Best CV score: {-random_search_rf.best_score_:.4f}")

# Grab the best model
best_model_random = random_search_rf.best_estimator_

{'regressor__n_estimators': [25, 50, 75, 100, 125], 'regressor__max_depth': [1, 2, 3, 4], 'regressor__min_samples_split': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x139469f90>}
Starting Random Search hyperparameter tuning...
Fitting 3 folds for each of 20 candidates, totalling 60 fits
[CV] END regressor__max_depth=1, regressor__min_samples_split=6, regressor__n_estimators=50; total time= 1.1min
[CV] END regressor__max_depth=1, regressor__min_samples_split=6, regressor__n_estimators=50; total time= 1.2min
[CV] END regressor__max_depth=1, regressor__min_samples_split=6, regressor__n_estimators=50; total time= 1.2min
[CV] END regressor__max_depth=1, regressor__min_samples_split=6, regressor__n_estimators=75; total time= 1.3min
[CV] END regressor__max_depth=1, regressor__min_samples_split=6, regressor__n_estimators=75; total time= 1.3min
[CV] END regressor__max_depth=1, regressor__min_samples_split=6, regressor__n_estimators=75; total time= 1.5min
[CV] END regressor_

In [38]:
y_pred_random_best = pipeline_best_rf.predict(X_test)
mae_random_best_rf = mean_absolute_error(y_test, y_pred_best)
#print(f"Baseline MAE:              ${mae_baseline:.2f}")
#print(f"Linear Regression MAE:     ${mae_linear:.2f}")
print(f"Random Forest MAE:         ${mae_rf:.2f}")
print(f"Tuned Random Forest MAE:   ${mae_best_rf:.2f}")
print(f"Random GridSearch best tuned Random Forest MAE:   ${mae_random_best_rf:.2f}")

Random Forest MAE:         $3.90
Tuned Random Forest MAE:   $3.32
Random GridSearch best tuned Random Forest MAE:   $3.32


Not any better, wider hyperparameter range/scope/space, and same MAE from the resulting best fit model.
15 minute runtime. Gridsearch was 18 combos to test. Here we defined 20, but we had a wider range of n_estimators and those take longer.

Same search space for Bayesian should be faster maybe?


In [ ]:
from skopt import BayesSearchCV
from skopt.space import Integer

    # 'regressor__n_estimators': list(range(25, 150, 25)),  # integers from 25-150 step 25
    # 'regressor__max_depth': list(range(1, 5, 1)),  # Samples integers from 1-5
    # 'regressor__min_samples_split': randint(2, 3, 4)  # Samples integers from 2-4

#The format is different for this, again
param_grid_bayesian = {
    'regressor__n_estimators': Integer(25, 150),
    'regressor__max_depth': Integer(1, 5),
    'regressor__min_samples_split': Integer(2, 4),
}

# Create BayesSearchCV
bayes_search_rf = BayesSearchCV(
    pipeline_rf,
    param_grid_bayesian,
    n_iter=30,
    cv=3,
    scoring='neg_mean_absolute_error',
    n_jobs=13,
    verbose=2,
    random_state=42,
    n_points=3
)

# Fit the search
print("\nStarting Bayesian Optimization hyperparameter tuning...")
bayes_search_rf.fit(X_train, y_train)

# Get best parameters / score
print(f"\nBest parameters: {bayes_search_rf.best_params_}")
print(f"Best CV score: {-bayes_search_rf.best_score_:.4f}")

# Grab the best model
pipeline_best_rf_bayes = bayes_search_rf.best_estimator_


Starting Bayesian Optimization hyperparameter tuning...
Fitting 3 folds for each of 3 candidates, totalling 9 fits
[CV] END regressor__max_depth=2, regressor__min_samples_split=3, regressor__n_estimators=63; total time= 1.0min
[CV] END regressor__max_depth=2, regressor__min_samples_split=3, regressor__n_estimators=63; total time= 1.0min
[CV] END regressor__max_depth=2, regressor__min_samples_split=3, regressor__n_estimators=63; total time= 1.0min
[CV] END regressor__max_depth=2, regressor__min_samples_split=3, regressor__n_estimators=77; total time= 1.1min
[CV] END regressor__max_depth=2, regressor__min_samples_split=3, regressor__n_estimators=77; total time= 1.1min
[CV] END regressor__max_depth=2, regressor__min_samples_split=3, regressor__n_estimators=77; total time= 1.1min
[CV] END regressor__max_depth=3, regressor__min_samples_split=3, regressor__n_estimators=142; total time= 1.7min
[CV] END regressor__max_depth=3, regressor__min_samples_split=3, regressor__n_estimators=142; total

/Users/jsg/repos/sbml/.venv/lib/python3.13/site-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [np.int64(5), np.int64(2), np.int64(25)] before, using random point [np.int64(5), np.int64(4), np.int64(124)]
  warnings.warn(
/Users/jsg/repos/sbml/.venv/lib/python3.13/site-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [np.int64(5), np.int64(2), np.int64(25)] before, using random point [np.int64(4), np.int64(3), np.int64(92)]
  warnings.warn(


Fitting 3 folds for each of 3 candidates, totalling 9 fits
[CV] END regressor__max_depth=4, regressor__min_samples_split=3, regressor__n_estimators=92; total time= 3.1min
[CV] END regressor__max_depth=4, regressor__min_samples_split=3, regressor__n_estimators=92; total time= 3.1min
[CV] END regressor__max_depth=4, regressor__min_samples_split=3, regressor__n_estimators=92; total time= 3.1min
[CV] END regressor__max_depth=5, regressor__min_samples_split=4, regressor__n_estimators=124; total time= 4.6min
[CV] END regressor__max_depth=5, regressor__min_samples_split=4, regressor__n_estimators=124; total time= 4.6min
[CV] END regressor__max_depth=5, regressor__min_samples_split=4, regressor__n_estimators=124; total time= 4.6min
[CV] END regressor__max_depth=5, regressor__min_samples_split=2, regressor__n_estimators=150; total time= 4.9min
[CV] END regressor__max_depth=5, regressor__min_samples_split=2, regressor__n_estimators=150; total time= 4.9min
[CV] END regressor__max_depth=5, regress

/Users/jsg/repos/sbml/.venv/lib/python3.13/site-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [np.int64(5), np.int64(2), np.int64(150)] before, using random point [np.int64(5), np.int64(4), np.int64(115)]
  warnings.warn(
/Users/jsg/repos/sbml/.venv/lib/python3.13/site-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [np.int64(4), np.int64(2), np.int64(150)] before, using random point [np.int64(5), np.int64(3), np.int64(101)]
  warnings.warn(


Fitting 3 folds for each of 3 candidates, totalling 9 fits


...

/Users/jsg/repos/sbml/.venv/lib/python3.13/site-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [np.int64(5), np.int64(2), np.int64(25)] before, using random point [np.int64(4), np.int64(3), np.int64(77)]
  warnings.warn(
/Users/jsg/repos/sbml/.venv/lib/python3.13/site-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [np.int64(5), np.int64(2), np.int64(25)] before, using random point [np.int64(4), np.int64(3), np.int64(89)]
  warnings.warn(
/Users/jsg/repos/sbml/.venv/lib/python3.13/site-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [np.int64(5), np.int64(2), np.int64(25)] before, using random point [np.int64(4), np.int64(3), np.int64(112)]
  warnings.warn(


Fitting 3 folds for each of 3 candidates, totalling 9 fits
[CV] END regressor__max_depth=4, regressor__min_samples_split=3, regressor__n_estimators=77; total time= 2.4min
[CV] END regressor__max_depth=4, regressor__min_samples_split=3, regressor__n_estimators=77; total time= 2.4min
[CV] END regressor__max_depth=4, regressor__min_samples_split=3, regressor__n_estimators=77; total time= 2.4min
[CV] END regressor__max_depth=4, regressor__min_samples_split=3, regressor__n_estimators=89; total time= 2.5min
[CV] END regressor__max_depth=4, regressor__min_samples_split=3, regressor__n_estimators=89; total time= 2.6min
[CV] END regressor__max_depth=4, regressor__min_samples_split=3, regressor__n_estimators=89; total time= 2.6min
[CV] END regressor__max_depth=4, regressor__min_samples_split=3, regressor__n_estimators=112; total time= 2.8min
[CV] END regressor__max_depth=4, regressor__min_samples_split=3, regressor__n_estimators=112; total time= 2.8min
[CV] END regressor__max_depth=4, regressor_

/Users/jsg/repos/sbml/.venv/lib/python3.13/site-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [np.int64(5), np.int64(2), np.int64(25)] before, using random point [np.int64(3), np.int64(4), np.int64(134)]
  warnings.warn(


Fitting 3 folds for each of 3 candidates, totalling 9 fits


Python(17169) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(17170) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
/Users/jsg/repos/sbml/.venv/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
Python(17175) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(17176) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(17177) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(17178) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(17179) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


**Never finishes, maybe my range is too big.**
This was just curiousity anyway, not part of the assignment.

In [ ]:
#Compare all three of these
y_pred_bayesian_best = pipeline_best_rf_bayes.predict(X_test)
mae_bayesian_best_rf = mean_absolute_error(y_test, y_pred_best)



print(f"Random Forest MAE:         ${mae_rf:.2f}")
print(f"Tuned Random Forest MAE:   ${mae_best_rf:.2f}")
print(f"Random GridSearch best tuned Random Forest MAE:   ${mae_random_best_rf:.2f}")
print(f"Bayesian GridSearch best tuned Random Forest MAE:   ${mae_bayesian_best_rf:.2f}")


# Conclusions
There is more to this than my quick experiements.

# AutoML / TPOT
We had lessons on these two tools and then never used them, maybe that is coming. I do suspect that I will need to stay local in PyCharm with larger datasets now for these toolsets - Collab cannot keep up without buying more CPU.